In [14]:
import gymnasium as gym
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.distributions import Categorical
import torch.nn.functional as F

Share Scaffolding

Policy Network

In [9]:
class PolicyNetwork(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, hidden_dims=[64, 64], activation=torch.nn.ReLU):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(torch.nn.Linear(prev_dim, hidden_dim))
            layers.append(activation())
            prev_dim = hidden_dim
        layers.append(torch.nn.Linear(prev_dim, output_dim))
        self.net = torch.nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

    def get_action(self, x):
        logits = self.net(x)
        action = torch.argmax(logits, dim=-1).item()
        return action

Value Network

In [13]:
class ValueNetwork(nn.Module):
    def __init__(self, state_dim, hidden_dim = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

Evaluate the Policy

In [10]:
def evaluate(policy: PolicyNetwork, env: gym.Env, n_episodes: int):
    with torch.no_grad():
        policy.eval()
        episode_returns = []
        for episode in range(n_episodes):
            obs, info = env.reset()
            done = False
            truncated = False
            episode_return = 0.0

            while not (done or truncated):
                obs_tensor = torch.FloatTensor(obs).unsqueeze(0)
                action = policy.get_action(obs_tensor)
                obs, reward, terminated, truncated, info = env.step(action)
                episode_return += float(reward)
            episode_returns.append(episode_return)

    mean_return = np.mean(episode_returns)
    return mean_return

make env

In [8]:
def make_env(env_id: str, seed: int):
    env = gym.make(env_id)
    env.action_space.seed(seed)
    env.observation_space.seed(seed)
    torch.manual_seed(seed)
    np.random.default_rng(seed)
    return env

SEC 1 - REINFORCE

In [12]:
def train_reinforce(env: gym.Env, policy: PolicyNetwork, gamma: float, batch_size: int, num_batches: int, optimizer: torch.optim.Adam):
    for batch in range(num_batches):
        batch_loss = 0.0

        for episode in range(batch_size):
            log_probs = []
            rewards = []
            state, _ = env.reset()
            done = False

            while not done:
                state_t = torch.FloatTensor(state)
                logits = policy(state_t)
                dist = Categorical(logits=logits)
                action = dist.sample()
                log_prob = dist.log_prob(action)
                next_state, reward, terminated, truncated, info = env.step(action)
                done = terminated or truncated
                log_probs.append(log_prob)
                rewards.append(reward)
                state = next_state

            returns = []
            G = 0.0
            for r in reversed(rewards):
                G = r + gamma * G
                returns.insert(0, G)

            returns = torch.tensor(returns)
            log_probs = torch.stack(log_probs)
            episode_loss = -(log_probs * returns).sum()
            batch_loss += episode_loss

        batch_loss = batch_loss / batch_size
        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()

REINFORCE with learned baseline

In [15]:
def train_reinforce_with_baseline(env: gym.Env, policy: PolicyNetwork, value: ValueNetwork, gamma: float, batch_size: int, num_batches: int, lr: float):
    policy_optimizer = torch.optim.Adam(policy.parameters(), lr=lr)
    value_optimizer = torch.optim.Adam(value.parameters(), lr=lr)

    for batch in range(num_batches):
        batch_loss = 0.0

        for episode in range(batch_size):
            log_probs = []
            rewards = []
            state, _ = env.reset()
            done = False

            while not done:
                state_t = torch.FloatTensor(state)
                logits = policy(state_t)
                dist = Categorical(logits=logits)
                action = dist.sample()
                log_prob = dist.log_prob(action)
                next_state, reward, terminated, truncated, info = env.step(action)
                done = terminated or truncated
                log_probs.append(log_prob)
                rewards.append(reward)
                state = next_state

            returns = []
            G = 0.0
            for r in reversed(rewards):
                G = r + gamma * G
                returns.insert(0, G)

            returns = torch.tensor(returns)
            states = torch.stack(states)
            log_probs = torch.stack(log_probs)
            values = value(states)
            advantages = returns - values.detach()

            policy_loss = -(log_probs * advantages).sum()
            value_loss = F.mse_loss(values, returns)

            policy_optimizer.zero_grad()
            policy_loss.backward()
            policy_optimizer.step()
            value_optimizer.zero_grad()
            value_loss.backward()
            value_optimizer.step()